# Cyclistic Bike-Share Analysis
This notebook downloads the last 12 months of Divvy/Cyclistic trip data, extracts it, merges it, cleans it, and prepares it for analysis.

In [4]:
import os
import pandas as pd
from datetime import datetime
import re

## 1. Ask

The main objective of this phase is to understand what the problem is. In this case we are trying to resolve the problem in regards to "How do annual members and casual riders use Cyclistic bikes differently?"

## 2. Prepare
In this phase we are looking at the methodlogy of how the data was collected and even where the data has been living/stored. We also need to look into whether the data is qualitative or quantative including how was collected to mitigate any bias. In this case the data has been collected by Motive International Inc whom has given us anonmized data to look at. 

## 3. Process



To process the data, it involves transforming raw data into actionable insights. I will use this process phase to filter, clean, remove, and explore the data that will be used to help identify trends and patterns in Part 4 Analyze. It will be sued to ensure data quality assurance throughout the entire process.

This process first begins by grabbing the data for each month from (October 2025 - September 2026). I will use pandas libaray to imporat each file and begin to merge them into one file for easier analysis. 

In [5]:
import pandas as pd

d1 = pd.read_csv("source/202510-divvy-tripdata.csv")
d2 = pd.read_csv("source/202511-divvy-tripdata.csv")
d3 = pd.read_csv("source/202512-divvy-tripdata.csv")
d4 = pd.read_csv("source/202601-divvy-tripdata.csv")
d5 = pd.read_csv("source/202602-divvy-tripdata.csv")
d6 = pd.read_csv("source/202603-divvy-tripdata.csv")
d7 = pd.read_csv("source/202604-divvy-tripdata.csv")
d8 = pd.read_csv("source/202605-divvy-tripdata.csv")
d9 = pd.read_csv("source/202606-divvy-tripdata.csv")
d10 = pd.read_csv("source/202607-divvy-tripdata.csv")
d11 = pd.read_csv("source/202608-divvy-tripdata.csv")
d12 = pd.read_csv("source/202609-divvy-tripdata.csv")

df = pd.concat([d1, d2, d3, d4, d5, d6, d7, d8, d9, d10, d11, d12])

df.shape

(6171785, 13)

We will then indentify the total number of columns and rows from this data set

In [6]:
df.shape

(6171785, 13)

In addition, we need to idnetify the columns and their respective types

In [10]:
df.info()

<class 'pandas.DataFrame'>
Index: 6171785 entries, 0 to 770561
Data columns (total 13 columns):
 #   Column              Dtype  
---  ------              -----  
 0   ride_id             str    
 1   rideable_type       str    
 2   started_at          str    
 3   ended_at            str    
 4   start_station_name  str    
 5   start_station_id    str    
 6   end_station_name    str    
 7   end_station_id      str    
 8   start_lat           float64
 9   start_lng           float64
 10  end_lat             float64
 11  end_lng             float64
 12  member_casual       str    
dtypes: float64(4), str(9)
memory usage: 659.2 MB


Check for any null values

In [7]:
df.isnull().sum()

ride_id                     0
rideable_type               0
started_at                  0
ended_at                    0
start_station_name    1289486
start_station_id      1289486
end_station_name      1355487
end_station_id        1355487
start_lat                   0
start_lng                   0
end_lat                  5451
end_lng                  5451
member_casual               0
dtype: int64

We need to drop these null values:

In [8]:
df = df.dropna()
df.isnull().sum()

ride_id               0
rideable_type         0
started_at            0
ended_at              0
start_station_name    0
start_station_id      0
end_station_name      0
end_station_id        0
start_lat             0
start_lng             0
end_lat               0
end_lng               0
member_casual         0
dtype: int64

Remove columns that are not needed for our analysis


In [11]:
df = df.drop(['start_station_name', 'start_station_id', 'end_station_name', 'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng'
], axis=1)

Modify the data type to better fit our needs

In [13]:
df["started_at"] = df["started_at"].astype('datetime64[ns]')
df["ended_at"] = df["ended_at"].astype('datetime64[ns]')

df.info()

<class 'pandas.DataFrame'>
Index: 4098022 entries, 0 to 770165
Data columns (total 5 columns):
 #   Column         Dtype         
---  ------         -----         
 0   ride_id        str           
 1   rideable_type  str           
 2   started_at     datetime64[ns]
 3   ended_at       datetime64[ns]
 4   member_casual  str           
dtypes: datetime64[ns](2), str(3)
memory usage: 219.8 MB


Create new columns to extract tge date, day, hour, and month

In [14]:
df['hour'] = df['started_at'].dt.hour
df['date'] = df['started_at'].dt.day
df['day'] = df['started_at'].dt.day_name()
df['month'] = df['started_at'].dt.month_name()

df.head()

,ride_id,rideable_type,started_at,ended_at,member_casual,hour,date,day,month
0,AAED255E1310E516,classic_bike,2025-10-27 08:19:53.208,2025-10-27 08:25:53.481,member,8,27,Monday,October
1,F90C86BF3BB85040,classic_bike,2025-10-14 07:42:55.465,2025-10-14 08:04:08.663,member,7,14,Tuesday,October
2,90429D9729A3EE8E,classic_bike,2025-10-12 08:18:23.613,2025-10-12 08:28:38.983,casual,8,12,Sunday,October
3,C8433F9B78A1E63D,electric_bike,2025-10-14 07:45:56.471,2025-10-14 08:07:24.447,member,7,14,Tuesday,October
4,62F3C413C442C154,electric_bike,2025-10-19 10:08:45.138,2025-10-19 10:12:34.771,member,10,19,Sunday,October


Sort and arrange the data properly

In [15]:
days_ordered = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday',
               'Saturday', 'Sunday']

df['day'] = pd.Categorical(df['day'], categories=days_ordered, ordered=True)

months_ordered = ['January', 'February', 'March', 'April', 'May', 
                 'June', 'July', 'August', 'September', 'October',
                 'November', 'December']

df['month'] = pd.Categorical(df['month'], categories=months_ordered, ordered=True)

Create a new column to store data that looks at the duration of the rides taken

In [16]:
df['ride_duration'] = (df['ended_at'] - df['started_at'])/pd.Timedelta(minutes=1)

df.head()

,ride_id,rideable_type,started_at,ended_at,member_casual,hour,date,day,month,ride_duration
0,AAED255E1310E516,classic_bike,2025-10-27 08:19:53.208,2025-10-27 08:25:53.481,member,8,27,Monday,October,6.004550
1,F90C86BF3BB85040,classic_bike,2025-10-14 07:42:55.465,2025-10-14 08:04:08.663,member,7,14,Tuesday,October,21.219967
2,90429D9729A3EE8E,classic_bike,2025-10-12 08:18:23.613,2025-10-12 08:28:38.983,casual,8,12,Sunday,October,10.256167
3,C8433F9B78A1E63D,electric_bike,2025-10-14 07:45:56.471,2025-10-14 08:07:24.447,member,7,14,Tuesday,October,21.466267
4,62F3C413C442C154,electric_bike,2025-10-19 10:08:45.138,2025-10-19 10:12:34.771,member,10,19,Sunday,October,3.827217


Look for any negative for ride duration

In [17]:
df[df['ride_duration'] < 0].count()

ride_id          19
rideable_type    19
started_at       19
ended_at         19
member_casual    19
hour             19
date             19
day              19
month            19
ride_duration    19
dtype: int64

Remove all negative durations

In [18]:
df = df[df['ride_duration'] > 0]
df[df['ride_duration'] < 0].count()

ride_id          0
rideable_type    0
started_at       0
ended_at         0
member_casual    0
hour             0
date             0
day              0
month            0
ride_duration    0
dtype: int64

In [20]:
df.to_csv("cyclistic_cleaned.csv")

Inspect the table data now

In [12]:
# Inspect the new table that has been created
print("\nList of column names:\n", df.columns)
print("\nHow many rows are in data frame?", len(df))
print("\nDimensions of the data frame:", df.shape)
print("\nSee the first 6 rows of data frame:\n", df.head())
print("\nSee list of columns and data types:\n")
df.info()
# Statistical summary of data (mainly for numerics)
print("\nStatistical summary of data:\n", df.describe())


List of column names:
 Index(['ride_id', 'rideable_type', 'started_at', 'ended_at', 'member_casual'],
      dtype='str')

How many rows are in data frame? 4098022

Dimensions of the data frame: (4098022, 5)

See the first 6 rows of data frame:
             ride_id  rideable_type               started_at  \
0  AAED255E1310E516   classic_bike  2025-10-27 08:19:53.208   
1  F90C86BF3BB85040   classic_bike  2025-10-14 07:42:55.465   
2  90429D9729A3EE8E   classic_bike  2025-10-12 08:18:23.613   
3  C8433F9B78A1E63D  electric_bike  2025-10-14 07:45:56.471   
4  62F3C413C442C154  electric_bike  2025-10-19 10:08:45.138   

                  ended_at member_casual  
0  2025-10-27 08:25:53.481        member  
1  2025-10-14 08:04:08.663        member  
2  2025-10-12 08:28:38.983        casual  
3  2025-10-14 08:07:24.447        member  
4  2025-10-19 10:12:34.771        member  

See list of columns and data types:

<class 'pandas.DataFrame'>
Index: 4098022 entries, 0 to 770165
Data columns (to

## 4. Analyze


## 5. Share